# 61 · Pathway analysis of human against mouse proximal tubule: whole PT, S1/S2/S3 and continuous methods compared

This notebook puts the pathway methods used in this project side by side, from the coarsest to the
finest view of the tubule:

1. **Whole PT.** One expression profile per specimen.
2. **Discrete.** One profile per specimen and segment (S1, S2, S3).
3. **Continuous.** Expression along the scFates coordinate (SCF13).

Each method gets a short description (the question it answers, the unit and statistic, the set test
and what it returns), followed by the code that produces its pathway calls. The comparison section
then shows how the calls overlap, how the statistics agree and what each family finds that the others
do not. Five worked pathways show one pathway score three ways.

**Assumptions and controls.**
- **The coordinate is treated as exact.** SCF13 is taken as a perfect reconstruction, so the notebook
  has no coordinate-sensitivity sections; those are notebooks 36, 57 and 58.
- **Specificity is checked by specimen relabeling**, which concerns specimens, not the coordinate.
  With 2 mouse and 2 human specimens, the two balanced mixed-species groupings should call nothing.
  A method is called **specific** when NCDR (mean relabeled calls ÷ species calls) is below 0.25 and
  each relabeling calls fewer than 5% of tested pathways (notebook 31's rule).
- **Cohort.** 2 control mice and 2 cortex sections from 1 human donor, all male. Results are
  descriptive.

**Sources.** Statistics that are expensive and already saved by notebooks 31, 37, 45/50 and 56 are
loaded and checked against their recorded values. Everything else is computed here from the shared
`pseudospace` modules. The protocol, including how the worked examples are chosen, was fixed in
`plan.md` (workstream B, notebook 61) before the notebook ran.

## 1 · Setup

In [ ]:
import argparse
import glob
import json
import os
import sys
from pathlib import Path

start = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
project = next(p for p in (start, *start.parents) if (p / 'pseudospace').is_dir())
sys.path.insert(0, str(project))
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument('--data-root', type=Path)
parser.add_argument('--results-root', type=Path)
args, _ = parser.parse_known_args()
data_root = Path(args.data_root or os.environ.get('PSEUDOSPACE_DATA_ROOT') or project / 'data').expanduser()
results_root = Path(args.results_root or os.environ.get('PSEUDOSPACE_RESULTS_ROOT') or project / 'results').expanduser()
nb12 = results_root / 'pt_pathway_remodeling_scfates'
nb31 = results_root / 'pt_pathway_method_selection'
nb37 = results_root / 'pt_pathway_final' / 'scfates'
nb45 = results_root / 'pt_revision_method' / 'tables'
nb50 = results_root / 'pt_primary_pathways' / 'tables'
nb56 = results_root / 'pt_pathway_continuous' / '56_taxonomy'
nb57 = results_root / 'pt_pathway_continuous' / '57_beyond_steps' / 'scf13' / 'tables'
output = results_root / 'pt_pathway_methods_compared'
tables, figures = output / 'tables', output / 'figures'
for folder in (tables, figures):
    folder.mkdir(parents=True, exist_ok=True)
REQUIRED = [results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv',
            nb31 / 'conventional_deseq2_all_partitions.csv', nb31 / 'conventional_signed_gsea_all_partitions.csv',
            nb31 / 'strategy_hits_all_partitions.csv', nb12 / 'conventional_pt_ora.csv',
            nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv', nb37 / 'tables' / 'table_S1_strategy_specificity.csv',
            nb37 / 'tables' / 'table_S2_pathways.csv', nb45 / 'anchor_our_cohort.csv', nb50 / 'cohort_step_screen_per_pathway.csv',
            nb56 / 'tables' / 'f_taxonomy.csv', nb56 / 'tables' / 'f_species_calls_by_approach.csv',
            nb57 / 'per_pathway_beyond_steps.csv']
for path in REQUIRED:
    if not path.is_file():
        raise FileNotFoundError(f'Missing input: {path} (run notebooks 12, 31, 37, 45, 50, 56 and 57 first)')

NOTEBOOK_LOGIC_VERSION = '61.methods_compared.1'  # Bump when a cached calculation changes.
ALPHA, NCDR_MAX, NULL_RATE = .05, .25, .05
LFC = 1.0                     # ORA: DE genes need |log2FC| >= 1 (notebook 12)
N_NULL, SEED = 9999, 12       # matched random sets (notebooks 12, 31, 37)
N_BINS, MIN_CELL = 8, 20      # split plot (notebook 37)
K_MODULES, KM_STARTS, KM_SEED, TOP_SHARE = 8, 20, 56, .10   # notebook 56
EXAMPLE_BINS = 12             # specimen x coordinate-bin means in the worked examples (notebook 41)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import sparse
from scipy.stats import hypergeom, spearmanr
from statsmodels.stats.multitest import multipletests

MM = 1 / 25.4
HUMAN, MOUSE, NULL, LIGHT, INK, MUTED, ACCENT = '#D55E00', '#0072B2', '#9A9A9A', '#D9D9D9', '#222222', '#666666', '#009E73'
SEGMENT_COLORS = {'S1': '#CDBCE3', 'S2': '#8A68BA', 'S3': '#3F2368'}
SPECIMENS = {'mouse': ['Ctrl1A2', 'Ctrl1A4'], 'human': ['HUK1_COR1', 'HUK1_MED1']}
SPECIMEN_MARKER = {'Ctrl1A2': 'o', 'Ctrl1A4': 's', 'HUK1_COR1': 'o', 'HUK1_MED1': 's'}
FAMILY_COLORS = {'whole PT': '#9A9A9A', 'discrete': '#8A68BA', 'bridge': '#CC79A7', 'continuous': '#009E73'}
plt.rcParams.update({
    'font.family': ['Liberation Sans', 'DejaVu Sans'], 'font.size': 6.5, 'axes.titlesize': 6.5,
    'axes.titleweight': 'bold', 'axes.labelsize': 6.5, 'xtick.labelsize': 6, 'ytick.labelsize': 6,
    'legend.fontsize': 6, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': .6, 'xtick.major.width': .6, 'ytick.major.width': .6, 'xtick.major.size': 2.5,
    'ytick.major.size': 2.5, 'lines.linewidth': 1.0, 'pdf.fonttype': 42, 'svg.fonttype': 'none', 'savefig.dpi': 300})
pd.set_option('display.width', 220, 'display.max_columns', 30, 'display.max_colwidth', 70)
print('Results folder:', output)

## 2 · Structures, genes, pathways and the specimen partitions

All methods use notebook 37's construction on SCF13:
- the 12,272 PT structures inside every specimen's 1st–99th percentile range of the coordinate;
- the 11,071 genes measured in both species and detected in ≥ 2% of structures;
- the 1,513 Reactome, Hallmark and KEGG pathways with 10–300 tested genes.

The **species split** puts the two human sections against the two mice. Each **relabeling** pairs one
human section with one mouse; in relabeled models the species effect is a nuisance term, so only
specimen differences remain to be found.

In [ ]:
from pseudospace.coordinate_inputs import pt_inputs, read_coordinate
from pseudospace.pathway_calibration import balanced_partitions, contrast_designs, nested_partial_f
from pseudospace.pathway_decoys import joint_matched_test
from pseudospace.pathway_remodeling import matched_pathway_tests
from pseudospace.split_plot import moderated_f
from pseudospace.stage_cache import cached_frame, cached_payload, digest
from pseudospace.stats_gam import gam_internal_knots, make_gam_design

inputs = pt_inputs(results_root, data_root, read_coordinate(results_root / 'pt_reconstruction_v2' / 'primary_pt_coordinate.csv'))
position, human, specimen, segment = (inputs[k] for k in ('position', 'human', 'specimen', 'segment'))
genes, gene_sets, strata = inputs['genes'], inputs['gene_sets'], inputs['strata']
pathways = list(gene_sets)
partitions = balanced_partitions(specimen, np.where(human, 'human', 'mouse'))
SWAPS = [p for p in partitions if p != 'species']
PARTS = ['species', *SWAPS]
assert len(position) == 12272 and len(genes) == 11071 and len(pathways) == 1513, 'notebook 37 universe changed'
table37 = pd.read_csv(nb37 / 'tables' / 'table_S2_pathways.csv').set_index('pathway_id')
NAMES = table37.name.reindex(pathways)
VIF = table37.vif.reindex(pathways)
spec37 = pd.read_csv(nb37 / 'tables' / 'table_S1_strategy_specificity.csv').set_index('strategy')
cache = output / 'stage_cache'
code = digest([NOTEBOOK_LOGIC_VERSION])
input_key = {'Y': digest(inputs['Y']), 'position': position, 'specimen': specimen, 'genes': genes}
print('Partitions:', partitions)

CALLS, EVIDENCE, DIRECTION, METHODS = {}, {}, {}, []
SHORT = {'W1': 'whole-PT GSEA', 'W2': 'whole-PT ORA', 'D1': 'S1/S2/S3 GSEA', 'D2': 'S1/S2/S3 ORA', 'D3': 'segment markers ORA',
         'D4': 'segment steps (joint)', 'C1': 'T_spatial joint (primary)', 'C2': 'T_spatial matched', 'C3': 'T_spatial GSEA',
         'C4': 'association per group', 'C5': 'start vs end', 'C6': 'activity, mean-z', 'C6b': 'activity, AUCell-like',
         'C7': 'curve modules + ORA', 'B1': 'six coordinate bins'}


def register(key, label, family, question, unit, set_test, returns, calls, evidence=None, direction=None, tested=None):
    """Record one method: calls per partition (sets), per-pathway evidence and direction for the species split."""
    CALLS[key] = {p: set(calls[p]) for p in PARTS if p in calls}
    if evidence is not None:
        EVIDENCE[key] = pd.Series(evidence, dtype=float).reindex(pathways)
    if direction is not None:
        DIRECTION[key] = pd.Series(direction, dtype=float).reindex(pathways)
    relabels = [len(CALLS[key][s]) for s in SWAPS if s in CALLS[key]]
    n_species, n_tested = len(CALLS[key]['species']), tested or len(pathways)
    ncdr = np.mean(relabels) / n_species if relabels and n_species else np.nan
    METHODS.append({'key': key, 'method': label, 'family': family, 'question': question, 'unit and statistic': unit,
                    'set test': set_test, 'returns': returns, 'tested': n_tested, 'species calls': n_species,
                    'relabeled calls': ', '.join(map(str, relabels)) if relabels else 'not applicable',
                    'NCDR': ncdr, 'specific': (bool(ncdr < NCDR_MAX and max(relabels) < NULL_RATE * n_tested)
                                               if relabels else None)})
    print(f"{key}: {n_species} species calls; relabeled {relabels or 'n/a'}")


def membership(sets, universe):
    index = pd.Index(universe)
    rows, cols = [], []
    for i, members in enumerate(sets.values()):
        idx = index.get_indexer(members)
        idx = idx[idx >= 0]
        rows.extend([i] * len(idx)), cols.extend(idx)
    return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(sets), len(index)))


def ora_pvalues(selected, matrix):
    """One-sided hypergeometric over-representation p for a boolean gene selection."""
    selected = np.asarray(selected, bool)
    if not selected.any():
        return np.ones(matrix.shape[0])
    k = np.asarray(matrix @ selected.astype(float)).ravel()
    return hypergeom.sf(k - 1, len(selected), np.asarray(matrix.sum(axis=1)).ravel(), selected.sum())


def joint_calls(statistics, stage):
    """Matched rank-AUC against 9,999 covariate-matched random sets, then the joint test (z ÷ √VIF)."""
    frame = cached_frame(stage, lambda: matched_pathway_tests(statistics, gene_sets, strata, n_null=N_NULL, seed=SEED),
                         root=cache, params={'n_null': N_NULL, 'seed': SEED},
                         inputs={'scores': statistics, 'sets': gene_sets, 'strata': strata}, code=code)
    tests = joint_matched_test(frame, VIF, keys=('statistic',))
    tests['called'] = tests.effect.gt(0) & tests.q_joint.le(ALPHA)
    return tests

## 3 · Whole-PT methods

The DESeq2 statistics are saved by notebook 31 for the species split and both relabelings. They
come from integer counts summed over each specimen. The species design is `~species`; a relabeled
design is `~species + group`.

In [ ]:
deseq = pd.read_csv(nb31 / 'conventional_deseq2_all_partitions.csv')
gsea = pd.read_csv(nb31 / 'conventional_signed_gsea_all_partitions.csv')
print(deseq.groupby(['partition', 'scope']).apply(lambda d: int(d.padj.le(ALPHA).sum()), include_groups=False)
      .unstack().rename_axis(columns='DESeq2 genes, padj ≤ 0.05'))

### W1 · Whole-PT pseudobulk DE with signed GSEA

| | |
|---|---|
| Question | Is the pathway, on average over the whole PT, shifted between species more than other genes? |
| Unit and statistic | specimen pseudobulk; DESeq2 Wald statistic, human against mouse |
| Set test | preranked GSEA (multilevel, weight 1), BH over pathways |
| Returns | yes/no and one direction (sign of NES) for the whole PT |

In [ ]:
whole = gsea[gsea.family.eq('whole_PT')]
register('W1', 'Whole-PT DE + signed GSEA', 'whole PT', 'average difference over the whole PT',
         'specimen pseudobulk; DESeq2 Wald', 'preranked GSEA', 'yes/no; one sign',
         {p: whole.pathway_id[whole.partition.eq(p) & whole.q.le(ALPHA)] for p in PARTS},
         evidence=-np.log10(whole[whole.partition.eq('species')].set_index('pathway_id').p.clip(lower=1e-300)),
         direction=np.sign(whole[whole.partition.eq('species')].set_index('pathway_id').NES), tested=whole.pathway_id.nunique())
row = spec37.loc['1 · Whole-PT pseudobulk + signed GSEA']   # Guard: notebook 37 Table S1
assert [len(CALLS['W1'][p]) for p in PARTS] == [row.species_calls, row.relabel_calls_Ctrl1A2, row.relabel_calls_Ctrl1A4]

### W2 · Whole-PT pseudobulk DE with ORA

| | |
|---|---|
| Question | Are the pathway's genes over-represented among genes that differ between species over the whole PT? |
| Unit and statistic | specimen pseudobulk; DESeq2 genes with BH ≤ 0.05 and \|log2FC\| ≥ 1, up and down separately |
| Set test | hypergeometric against all tested genes; BH over pathway × direction; called if either direction passes |
| Returns | yes/no and direction (up or down list) for the whole PT |

In [ ]:


def deseq_ora(frame, scopes):
    """ORA p-values per scope and direction (pathways x tests) on the tested genes of each scope."""
    columns, labels = [], []
    for scope in scopes:
        rows = frame[frame.scope.eq(scope)].dropna(subset=['stat']).set_index('gene')
        matrix = membership(gene_sets, list(rows.index))
        for direction in (1, -1):
            selected = rows.padj.le(ALPHA).to_numpy() & (direction * rows.log2FoldChange).ge(LFC).to_numpy()
            columns.append(ora_pvalues(selected, matrix))
            labels.append((scope, direction))
    return pd.DataFrame(np.column_stack(columns), index=pathways, columns=pd.MultiIndex.from_tuples(labels))


def ora_summary(pvalues):
    q = pd.DataFrame(multipletests(pvalues.to_numpy().ravel(), method='fdr_bh')[1].reshape(pvalues.shape),
                     index=pvalues.index, columns=pvalues.columns)
    best = pvalues.idxmin(axis=1)
    return q.le(ALPHA).any(axis=1), -np.log10(pvalues.min(axis=1).clip(lower=1e-300)), best.map(lambda c: c[1])


ora = {}
for family, scopes in (('whole', ['whole_PT']), ('segments', ['PT-S1', 'PT-S2', 'PT-S3'])):
    for p in PARTS:
        ora[(family, p)] = ora_summary(deseq_ora(deseq[deseq.partition.eq(p)], scopes))
register('W2', 'Whole-PT DE + ORA', 'whole PT', 'average difference, thresholded genes',
         'specimen pseudobulk; DESeq2 DE gene lists', 'hypergeometric ORA', 'yes/no; up or down',
         {p: [x for x, h in ora[('whole', p)][0].items() if h] for p in PARTS},
         evidence=ora[('whole', 'species')][1], direction=ora[('whole', 'species')][2])
saved56 = pd.read_csv(nb56 / 'tables' / 'f_species_calls_by_approach.csv', index_col=0)
assert CALLS['W2']['species'] == set(saved56.index[saved56['Whole-PT pseudobulk DE + ORA']])  # Guard: notebook 56

## 4 · Discrete methods (S1/S2/S3)

### D1 · S1/S2/S3 pseudobulk DE with signed GSEA

| | |
|---|---|
| Question | Within each segment, is the pathway shifted between species more than other genes? |
| Unit and statistic | specimen × segment pseudobulk; DESeq2 Wald per segment |
| Set test | preranked GSEA per segment; BH over pathway × segment; called if any segment passes |
| Returns | yes/no and a sign per segment, so a three-level location |

In [ ]:
segs = gsea[gsea.family.eq('segments')]
best_segment = segs[segs.partition.eq('species')].sort_values('p').drop_duplicates('pathway_id').set_index('pathway_id')
register('D1', 'S1/S2/S3 DE + signed GSEA', 'discrete', 'difference within a segment',
         'specimen × segment pseudobulk; DESeq2 Wald', 'preranked GSEA per segment', 'yes/no; sign per segment',
         {p: segs.pathway_id[segs.partition.eq(p) & segs.q.le(ALPHA)].unique() for p in PARTS},
         evidence=-np.log10(best_segment.p.clip(lower=1e-300)), direction=np.sign(best_segment.NES),
         tested=segs.pathway_id.nunique())
row = spec37.loc['2 · S1/S2/S3 pseudobulk + signed GSEA']
assert [len(CALLS['D1'][p]) for p in PARTS] == [row.species_calls, row.relabel_calls_Ctrl1A2, row.relabel_calls_Ctrl1A4]

### D2 · S1/S2/S3 pseudobulk DE with ORA

| | |
|---|---|
| Question | Within a segment, are the pathway's genes over-represented among species-DE genes? |
| Unit and statistic | specimen × segment pseudobulk; DESeq2 DE gene lists per segment and direction |
| Set test | hypergeometric; BH over pathway × segment × direction; called if any test passes |
| Returns | yes/no; segment and direction of the strongest test |

In [ ]:
register('D2', 'S1/S2/S3 DE + ORA', 'discrete', 'over-representation within a segment',
         'specimen × segment pseudobulk; DESeq2 DE gene lists', 'hypergeometric ORA', 'yes/no; segment and direction',
         {p: [x for x, h in ora[('segments', p)][0].items() if h] for p in PARTS},
         evidence=ora[('segments', 'species')][1], direction=ora[('segments', 'species')][2])
assert CALLS['D2']['species'] == set(saved56.index[saved56['S1/S2/S3 pseudobulk DE + ORA']])  # Guard: notebook 56

### D3 · Segment markers with ORA

| | |
|---|---|
| Question | Which pathways characterise S1, S2 or S3 (species pooled)? This is a zonation question, not a species comparison |
| Unit and statistic | specimen × segment pseudobulk; DESeq2 segment against the rest, species as covariate (notebook 12) |
| Set test | hypergeometric on marker genes (BH ≤ 0.05, \|log2FC\| ≥ 1) per segment and direction; BH over the family |
| Returns | the segments a pathway marks; no species information, so there is no relabeling control |

In [ ]:
markers = pd.read_csv(nb12 / 'conventional_pt_ora.csv')
markers = markers[markers.scope.eq('segment_vs_rest')].copy()
np.testing.assert_allclose(multipletests(markers.p_value, method='fdr_bh')[1], markers.q_family, rtol=1e-9)  # Guard
best_marker = markers.sort_values('p_value').drop_duplicates('pathway_id').set_index('pathway_id')
register('D3', 'Segment markers + ORA', 'discrete', 'what characterises each segment (species pooled)',
         'specimen × segment pseudobulk; segment vs rest', 'hypergeometric ORA', 'segment identity',
         {'species': markers.pathway_id[markers.q_family.le(ALPHA)].unique()},
         evidence=-np.log10(best_marker.p_value.clip(lower=1e-300)), tested=markers.pathway_id.nunique())
print('Segments marked:', markers[markers.q_family.le(ALPHA)].groupby(['segment_class', 'direction']).size().to_dict())

### D4 · Coordinate-free segment-step joint screen

| | |
|---|---|
| Question | Does the pathway's S1 → S2 → S3 profile differ in shape between species, beyond a species offset? |
| Unit and statistic | structures; per gene the partial F of species × segment steps beyond a species offset, with segment intercepts as the shared shape (no coordinate) |
| Set test | matched rank-AUC against covariate-matched random sets, divided by √VIF (joint test) |
| Returns | yes/no; a step pattern (2 df) |

Notebook 45 ran this screen and calibrated it on many external 2 + 2 donor draws. Notebook 50 saved
the per-pathway result for our cohort, and notebook 45 the relabeled call counts.

In [ ]:
steps = pd.read_csv(nb50 / 'cohort_step_screen_per_pathway.csv').set_index('pathway_id')
anchor = pd.read_csv(nb45 / 'anchor_our_cohort.csv').set_index('strategy').loc['P1 T_steps joint']
assert int(steps.called.sum()) == int(anchor.species) == 50                    # Guard: notebooks 45 and 50
# Notebook 45 saved only the relabeled counts; both are zero, so the relabeled call sets are empty.
assert int(anchor.relabel_Ctrl1A2) == 0 and int(anchor.relabel_Ctrl1A4) == 0
register('D4', 'Segment-step screen, coordinate-free (joint)', 'discrete', 'does the segment profile differ',
         'structures; species × segment steps (partial F)', 'matched rank-AUC ÷ √VIF', 'yes/no; step pattern',
         {'species': steps.index[steps.called.astype(bool)], SWAPS[0]: [], SWAPS[1]: []}, evidence=steps.z_joint)

## 5 · Continuous methods on SCF13

Gene-level models are notebook 31's nested weighted regression splines (6 df) along the coordinate,
with specimen contrasts and equal weight per specimen. The positional statistic is
**T_spatial**: the partial F of species × position beyond a species offset. Notebook 37 saved the
matched tests for every statistic and partition.

In [ ]:
tests37 = pd.read_csv(nb37 / 'tables' / 'matched_and_joint_tests_all_partitions.csv')
tests37 = tests37[tests37.partition.isin(PARTS)]
recomputed = joint_matched_test(tests37.drop(columns=['z_matched', 'vif', 'z_joint', 'p_joint', 'q_joint']), VIF)
np.testing.assert_allclose(recomputed.z_joint, tests37.z_joint, rtol=1e-9)   # Guard: joint z = matched z ÷ √VIF


def from37(statistic, column):
    block = tests37[tests37.statistic.eq(statistic)]
    return {p: block.pathway_id[block.partition.eq(p) & block.effect.gt(0) & block[column].le(ALPHA)] for p in PARTS}, \
        block[block.partition.eq('species')].set_index('pathway_id')

### C1 · T_spatial with the joint test (the primary method)

| | |
|---|---|
| Question | Is the positional shape of the pathway's genes more species-dependent than for matched other genes? |
| Unit and statistic | structures along the coordinate; T_spatial per gene |
| Set test | one-sided rank-AUC against 9,999 random sets matched on expression, detection and coverage; z ÷ √VIF from within-specimen residual correlations; BH |
| Returns | yes/no, plus the pathway's divergence curve D(s), where it peaks and its direction there |

In [ ]:
calls, species_rows = from37('T_spatial', 'q_joint')
register('C1', 'T_spatial, joint test (primary)', 'continuous', 'species-dependent positional shape',
         'structures; T_spatial (partial F)', 'matched rank-AUC ÷ √VIF', 'yes/no; D(s), peak, direction at peak',
         calls, evidence=species_rows.z_joint)
row = spec37.loc['8 · Gene GAM × smooth position (T_spatial) (joint test)']
assert [len(CALLS['C1'][p]) for p in PARTS] == [row.species_calls, row.relabel_calls_Ctrl1A2, row.relabel_calls_Ctrl1A4]

### C2 · T_spatial with the matched rank-AUC only

Same statistic and null, without the correlation adjustment: genes are treated as independent, so
the test is anti-conservative when members are correlated (notebook 37).

In [ ]:
calls, species_rows = from37('T_spatial', 'q_normal')
register('C2', 'T_spatial, matched rank-AUC', 'continuous', 'as C1, genes treated as independent',
         'structures; T_spatial', 'matched rank-AUC', 'yes/no', calls, evidence=species_rows.z_matched)
row = spec37.loc['8 · Gene GAM × smooth position (T_spatial)']
assert [len(CALLS['C2'][p]) for p in PARTS] == [row.species_calls, row.relabel_calls_Ctrl1A2, row.relabel_calls_Ctrl1A4]

### C3 · T_spatial with preranked GSEA

The same gene ranking tested with GSEA (weight 0) instead of matched random sets.

In [ ]:
hits31 = pd.read_csv(nb31 / 'strategy_hits_all_partitions.csv')
c3 = hits31[hits31.strategy.eq('8c · T_spatial, preranked GSEA')]
spatial_gsea_file = sorted(glob.glob(str(nb37 / 'stage_cache' / 'spatial_gsea_species__*.csv')))[-1]
spatial_gsea = pd.read_csv(spatial_gsea_file).set_index('pathway_id')
register('C3', 'T_spatial, preranked GSEA', 'continuous', 'as C1, GSEA null', 'structures; T_spatial',
         'preranked GSEA', 'yes/no', {p: c3.pathway_id[c3.partition.eq(p) & c3.hit] for p in PARTS},
         evidence=-np.log10(spatial_gsea.p.clip(lower=1e-300)))
row = spec37.loc['8c · T_spatial, preranked GSEA']
assert [len(CALLS['C3'][p]) for p in PARTS] == [row.species_calls, row.relabel_calls_Ctrl1A2, row.relabel_calls_Ctrl1A4]

### Group-difference curves Δ(s)

C4–C7 use each partition's fitted group-difference curve Δ(s) on 61 grid points (notebook 31's full
design; relabelings keep the species shape as a nuisance term). Notebook 56 cached these; they are
loaded and their `T_spatial` is checked against notebook 37.

In [ ]:


def newest(folder, stage):
    files = glob.glob(str(folder / f'{stage}__*.npz'))
    files.sort(key=lambda p: json.loads(Path(p + '.key.json').read_text())['written_utc'])
    with np.load(files[-1]) as stored:
        return {k: stored[k] for k in stored.files}


curves = {p: newest(nb56 / 'stage_cache', 'curves_' + p.replace(':', '_').replace('+', '_')) for p in PARTS}
gene37 = pd.read_csv(nb37 / 'tables' / 'gene_level_statistics.csv', index_col=0).T_spatial.reindex(genes)
np.testing.assert_allclose(curves['species']['T_spatial'], gene37, rtol=1e-6)   # Guard
grid = inputs['grid']
knots = gam_internal_knots(position, basis_df=6)

### C4 · tradeSeq-style association, compared between groups

| | |
|---|---|
| Question | Which pathways are zonated (position-dependent) in one species but not detectably in the other? This mirrors running tradeSeq's `associationTest` per condition and comparing the lists |
| Unit and statistic | structures of one group; per gene the partial F of a 6-df spline against specimen offsets |
| Set test | joint test per group; a pathway is "differential" when it is called in exactly one group |
| Returns | zonated-in-one-group lists; no direct test of a difference |

This is the "difference in significance" design. It is included because it is a common way to read
per-condition trajectory tests, and the relabelings show what it calls when there is no species
difference.

In [ ]:


def association_statistics(rows):
    s, spec = position[rows], specimen[rows]
    names = np.unique(spec)
    nuisance = np.column_stack([(spec == n).astype(float) - (spec == names[-1]).astype(float) for n in names[:-1]])
    designs = {'offset': np.column_stack([np.ones(rows.sum()), nuisance]),
               'spline': np.column_stack([make_gam_design(s, knots), nuisance])}
    return nested_partial_f(inputs['Y'][rows], designs, np.ones(rows.sum()), {'F': ('offset', 'spline')})['F']


association_calls, association_z = {}, {}
for p in PARTS:
    groups = {'group 1': np.isin(specimen, partitions[p])}   # species split: group 1 = the human sections
    groups['group 0'] = ~groups['group 1']
    called = {}
    for g, rows in groups.items():
        stat = pd.DataFrame(cached_payload(f"association_{p.replace(':', '_').replace('+', '_')}_{g.replace(' ', '')}",
                                           lambda rows=rows: {'F': association_statistics(rows)}, root=cache,
                                           params={'partition': p, 'group': g}, inputs=input_key, code=code), index=genes)
        tests = joint_calls(stat.rename(columns={'F': 'association'}),
                            f"association_tests_{p.replace(':', '_').replace('+', '_')}_{g.replace(' ', '')}")
        called[g] = set(tests.pathway_id[tests.called])
        if p == 'species':
            association_z[g] = tests.set_index('pathway_id').z_joint
    association_calls[p] = called['group 1'] ^ called['group 0']
    print(f'{p}: zonated in group 1 only {len(called["group 1"] - called["group 0"])}, '
          f'group 0 only {len(called["group 0"] - called["group 1"])}, both {len(called["group 1"] & called["group 0"])}')
register('C4', 'tradeSeq-style association per group, compared', 'continuous', 'zonated in one species only',
         'structures of one group; spline vs offset (partial F)', 'joint test per group; called in one group only',
         'zonated-in-one-group lists', association_calls,
         evidence=(association_z['group 1'] - association_z['group 0']).abs(),
         direction=np.sign(association_z['group 1'] - association_z['group 0']))

### C5 · tradeSeq-style start versus end

| | |
|---|---|
| Question | Does the species difference at the end of the PT differ from that at the start (`startVsEndTest` applied to the difference curve)? |
| Unit and statistic | per gene \|Δ(end) − Δ(start)\| from the fitted group-difference curve |
| Set test | joint test (matched rank-AUC ÷ √VIF) |
| Returns | yes/no for an early-against-late change; nothing about the middle |

In [ ]:
c5 = {p: joint_calls(pd.DataFrame({'start_vs_end': np.abs(curves[p]['delta'][:, -1] - curves[p]['delta'][:, 0])},
                                  index=genes), f"start_vs_end_{p.replace(':', '_').replace('+', '_')}") for p in PARTS}
register('C5', 'tradeSeq-style start vs end (joint)', 'continuous', 'early-versus-late change of the species difference',
         'structures; |Δ(end) − Δ(start)|', 'matched rank-AUC ÷ √VIF', 'yes/no; early vs late only',
         {p: c5[p].pathway_id[c5[p].called] for p in PARTS}, evidence=c5['species'].set_index('pathway_id').z_joint)

### C6 · Pathway-activity curves, tested at specimen level

| | |
|---|---|
| Question | Does the pathway's activity curve along the PT differ in shape between species, judged against specimen-to-specimen variation? This is a self-contained test: no comparison with other genes |
| Unit and statistic | per structure a pathway score (mean of species-balanced z-scored member expression; also an AUCell-like top-5% rank score), averaged in specimen × 8 coordinate-bin cells |
| Set test | split plot: cubic species × position against specimen × position; moderated F; BH |
| Returns | an activity curve per specimen; yes/no |

Notebook 56 cached the structure scores. The split plot is recomputed here.

In [ ]:
scores = newest(nb56 / 'stage_cache', 'structure_scores')
lo, hi = grid[0], grid[-1]
bins = np.clip(((position - lo) / (hi - lo) * N_BINS).astype(int), 0, N_BINS - 1)
cell_sizes = pd.Series(1, index=pd.MultiIndex.from_arrays([specimen, bins])).groupby(level=[0, 1]).size()
cell_keys = cell_sizes[cell_sizes >= MIN_CELL].index
ALL_SPECIMENS = SPECIMENS['mouse'] + SPECIMENS['human']


def split_plot(matrix):
    means = pd.DataFrame(matrix).groupby([specimen, bins]).mean().loc[cell_keys]
    cell_spec = means.index.get_level_values(0).to_numpy()
    basis = make_gam_design((means.index.get_level_values(1).to_numpy() + .5) / N_BINS, np.array([]))[:, 1:]
    spec_cols = (cell_spec[:, None] == np.array(ALL_SPECIMENS)[None, :]).astype(float)
    cell_human = np.isin(cell_spec, SPECIMENS['human']).astype(float)
    out = {}
    for p in PARTS:
        if p == 'species':
            design = np.column_stack([spec_cols, basis, cell_human[:, None] * basis])
        else:
            group = np.isin(cell_spec, partitions[p]).astype(float)
            design = np.column_stack([spec_cols, basis, cell_human[:, None] * basis, group[:, None] * basis])
        test = list(range(design.shape[1] - basis.shape[1], design.shape[1]))
        out[p] = moderated_f(means.to_numpy(float), np.ones(means.shape), design, test)['p']
    return out


activity = {}
for score_name, key, label in (('mean_z', 'C6', 'mean-z module score'), ('aucell', 'C6b', 'AUCell-like score')):
    pvalues = split_plot(scores[score_name])
    activity[key] = pvalues['species']
    register(key, f'Pathway activity ({label}), split plot', 'continuous', 'activity curve shape differs (self-contained)',
             f'specimen × bin means of a {label}', 'split-plot moderated F', 'activity curve per specimen; yes/no',
             {p: np.array(pathways)[multipletests(pv, method='fdr_bh')[1] <= ALPHA] for p, pv in pvalues.items()},
             evidence=pd.Series(-np.log10(np.clip(pvalues['species'], 1e-300, None)), index=pathways))
taxonomy56 = pd.read_csv(nb56 / 'tables' / 'f_taxonomy.csv').set_index('approach')
for key, label in (('C6', 'Pathway activity (mean-z module score), specimen × bin split plot'),
                   ('C6b', 'Pathway activity (AUCell-like score), specimen × bin split plot')):
    assert len(CALLS[key]['species']) == int(taxonomy56.loc[label, 'species calls'])   # Guard: notebook 56

### C7 · Curve modules with ORA

| | |
|---|---|
| Question | Which shapes of species difference recur among strongly divergent genes, and which pathways fill each shape? |
| Unit and statistic | the top 10% of genes by T_spatial; Δ(s) centred and scaled; k-means (k = 8, chosen by resampling stability in notebook 56) |
| Set test | hypergeometric ORA per module against all tested genes; BH over module × pathway |
| Returns | module shapes (centroids) and the pathways enriched in each |

In [ ]:
from sklearn.cluster import KMeans

genes_index = np.arange(len(genes))
module_matrix = membership(gene_sets, genes)
module_calls, module_p, centroids = {}, None, None
for p in PARTS:
    top = np.sort(np.argsort(-curves[p]['T_spatial'])[:int(round(TOP_SHARE * len(genes)))])
    d = curves[p]['delta'][top]
    d = (d - d.mean(axis=1, keepdims=True)) / np.maximum(d.std(axis=1, keepdims=True), 1e-12)
    model = KMeans(n_clusters=K_MODULES, n_init=KM_STARTS, random_state=KM_SEED).fit(d)
    columns = []
    for m in range(K_MODULES):
        selected = np.zeros(len(genes), bool)
        selected[top[model.labels_ == m]] = True
        columns.append(ora_pvalues(selected, module_matrix))
    pv = np.column_stack(columns)
    q = multipletests(pv.ravel(), method='fdr_bh')[1].reshape(pv.shape)
    module_calls[p] = np.array(pathways)[(q <= ALPHA).any(axis=1)]
    if p == 'species':
        module_p, centroids, module_genes = pv, model.cluster_centers_, (top, model.labels_)
register('C7', f'Curve modules (k = {K_MODULES}) + ORA', 'continuous', 'which difference shapes recur',
         'top-10% genes; k-means on standardised Δ(s)', 'hypergeometric ORA per module', 'module shapes; member pathways',
         module_calls, evidence=pd.Series(-np.log10(module_p.min(axis=1).clip(1e-300)), index=pathways))
assert len(CALLS['C7']['species']) == int(taxonomy56.loc[f'Curve modules (k = {K_MODULES}) + ORA', 'species calls'])  # Guard
assert [len(CALLS['C7'][s]) for s in SWAPS] == [int(x) for x in
                                                 taxonomy56.loc[f'Curve modules (k = {K_MODULES}) + ORA', 'relabeled calls'].split(', ')]

### B1 · Six equal coordinate bins, the bridge between discrete and continuous

| | |
|---|---|
| Question | Does the species profile across six equal bins of the coordinate differ beyond an offset? |
| Unit and statistic | structures; species × 6-bin steps beyond a species offset (partial F) |
| Set test | joint test |
| Returns | yes/no; a six-level profile |

In [ ]:
calls, species_rows = from37('T_bins_spatial', 'q_joint')
register('B1', 'Six coordinate bins (joint)', 'bridge', 'does the 6-bin profile differ', 'structures; species × 6 bins',
         'matched rank-AUC ÷ √VIF', 'yes/no; six-level profile', calls, evidence=species_rows.z_joint)
row = spec37.loc['7 · Gene GAM × 6 equal bins (joint test)']
assert [len(CALLS['B1'][p]) for p in PARTS] == [row.species_calls, row.relabel_calls_Ctrl1A2, row.relabel_calls_Ctrl1A4]

## 6 · Comparison

### 6.0 · Collapsing redundant pathways into programs

Reactome, KEGG and Hallmark contain many near-duplicate pathways, so a call count partly counts
library redundancy.

**Primary rule (notebook 37, SM17).** Pathways are grouped by the genes that carry their signal:
- those genes are the members in the top 10% of genes by T_spatial, or all members when fewer than
  three qualify;
- grouping is average linkage on 1 − overlap coefficient, cut at 0.5;
- it uses notebook 37's function;
- **order-invariant** (plan addendum 3): average linkage with tied distances depends on input
  order, so pathways are sorted by `pathway_id` before clustering.

The 36 robust pathways are regrouped and compared with notebook 37's 17 programs, which were made in
library order and are superseded by this rule.

**Membership check.** The same linkage and cut on 1 − Jaccard of full membership. The rules were fixed
in `plan.md` before this ran.

In [ ]:
from scipy.cluster.hierarchy import fcluster, linkage
from scipy.spatial.distance import squareform
from sklearn.metrics import adjusted_rand_score
from pseudospace.pathway_decoys import overlap_programs

PROGRAM_CUT = .5
contributing = set(gene37.index[gene37.ge(gene37.quantile(1 - TOP_SHARE))])


def driver_programs(ids):
    """Order-invariant programs: notebook 37's rule with pathways sorted by pathway_id before clustering.

    Average linkage with tied distances depends on input order; a fixed (sorted) order makes the result
    canonical. Programs are numbered by size, ties by the smallest member id.
    """
    ids = sorted(set(ids))
    if not ids:
        return pd.Series(dtype=int)
    return overlap_programs(gene_sets, ids, contributing=contributing, cut=PROGRAM_CUT).set_index('pathway_id').program


def membership_programs(ids):
    """The same linkage and cut on 1 - Jaccard of full membership (a check), sorted by pathway_id."""
    ids = sorted(set(ids))
    if len(ids) < 2:
        return pd.Series(1, index=ids, dtype=int)
    members = [set(gene_sets[p]) for p in ids]
    d = np.zeros((len(ids), len(ids)))
    for i in range(len(ids)):
        for j in range(i):
            d[i, j] = d[j, i] = 1 - len(members[i] & members[j]) / len(members[i] | members[j])
    raw = fcluster(linkage(squareform(d), 'average'), PROGRAM_CUT, criterion='distance')
    relabel = {old: new for new, old in enumerate(pd.Series(raw).value_counts().index, start=1)}
    return pd.Series([relabel[r] for r in raw], index=ids)


def partition(labels):
    return {frozenset(block.index) for _, block in labels.groupby(labels)}


robust36 = table37.index[table37.robust.astype(bool)]
library_order = overlap_programs(gene_sets, [p for p in pathways if p in set(robust36)], contributing=contributing,
                                 cut=PROGRAM_CUT).set_index('pathway_id').program
assert partition(library_order) == partition(table37.loc[robust36, 'program'].astype(int))   # Guard: notebook 37's rule
invariant36 = driver_programs(robust36)
old37 = table37.loc[robust36, 'program'].astype(int)


def comembers(labels, p):
    return frozenset(labels.index[labels.eq(labels[p])]) - {p}


moved = [p for p in sorted(robust36) if comembers(invariant36, p) != comembers(old37, p)]
MOVES = pd.DataFrame({'pathway': NAMES[moved], 'notebook 37 program': old37[moved],
                      'notebook 37 co-members': [', '.join(sorted(NAMES[list(comembers(old37, p))])) for p in moved],
                      'order-invariant program': invariant36[moved],
                      'order-invariant co-members': [', '.join(sorted(NAMES[list(comembers(invariant36, p))])) for p in moved]})
MOVES.to_csv(tables / 'robust36_program_moves.csv')
print(f'Robust 36: notebook 37 (library order) {old37.nunique()} programs; order-invariant {invariant36.nunique()} programs; '
      f'{len(moved)} pathways change co-members.')
display(MOVES)


def program_names(labels, score):
    """Name each program after its member with the highest score (ties by pathway id)."""
    names = {}
    for program, block in labels.groupby(labels):
        best = score.reindex(block.index).fillna(-np.inf).sort_index().sort_values(ascending=False, kind='stable').index[0]
        names[program] = NAMES[best]
    return labels.map(names)


def disagreements(ids):
    a, b = driver_programs(ids), membership_programs(ids)
    rows = []
    for i, p in enumerate(sorted(ids)):
        for q in sorted(ids)[i + 1:]:
            same_a, same_b = a[p] == a[q], b[p] == b[q]
            if same_a != same_b:
                rows.append({'pathway 1': NAMES[p], 'pathway 2': NAMES[q],
                             'grouped by drivers': same_a, 'grouped by membership': same_b,
                             'contributing overlap coefficient': len(set(gene_sets[p]) & set(gene_sets[q]) & contributing)
                             / max(min(len(set(gene_sets[p]) & contributing), len(set(gene_sets[q]) & contributing)), 1),
                             'membership Jaccard': len(set(gene_sets[p]) & set(gene_sets[q]))
                             / len(set(gene_sets[p]) | set(gene_sets[q]))})
    return pd.DataFrame(rows), adjusted_rand_score(a.sort_index(), b.sort_index())


primary_list = pd.read_csv(nb50 / 'primary_pathway_list.csv')
primary_ids = primary_list.pathway_id.tolist()
PROGRAM_SETS = {'primary 22': primary_ids, 'C1 calls (60)': sorted(CALLS['C1']['species'])}
program_summary = []
for label, ids in PROGRAM_SETS.items():
    drivers, members = driver_programs(ids), membership_programs(ids)
    table_dis, ari = disagreements(ids)
    table_dis.to_csv(tables / f"program_rule_disagreements_{label.split(' ')[0].lower()}.csv", index=False)
    program_summary.append({'set': label, 'pathways': len(ids), 'programs (driver rule)': drivers.nunique(),
                            'programs (membership check)': members.nunique(),
                            'pathway pairs grouped differently': len(table_dis), 'adjusted Rand index': ari})
    print(f'{label}: pairs grouped differently by the two rules')
    display(table_dis.round(2))
display(pd.DataFrame(program_summary).round(2))

primary_drivers = driver_programs(primary_ids)
primary_table = primary_list.copy()
primary_table['program'] = primary_table.pathway_id.map(primary_drivers)
primary_table['program name'] = primary_table.pathway_id.map(program_names(primary_drivers, EVIDENCE['C1']))
primary_table['membership-check program'] = primary_table.pathway_id.map(membership_programs(primary_ids))
primary_table.to_csv(tables / 'primary22_programs.csv', index=False)
c1_drivers = driver_programs(CALLS['C1']['species'])
c1_programs = pd.DataFrame({'program': c1_drivers, 'program name': program_names(c1_drivers, EVIDENCE['C1']),
                            'pathway': NAMES[c1_drivers.index]})
c1_programs.to_csv(tables / 'c1_calls_programs.csv')
display(primary_table[['name', 'program', 'program name', 'membership-check program']].sort_values('program'))

### 6.1 · One table

"Specific" applies notebook 31's rule. Segment markers (D3) answer a zonation question, so they have
no species calls to relabel. The program columns collapse each method's own species calls with the
rules of section 6.0. The driver rule uses the same top-decile T_spatial genes for every method, so
both columns measure library redundancy on one scale.

In [ ]:
METHOD_TABLE = pd.DataFrame(METHODS).set_index('key')
METHOD_TABLE['∩ C1 (primary)'] = [len(CALLS[k]['species'] & CALLS['C1']['species']) for k in METHOD_TABLE.index]
METHOD_TABLE['programs (driver rule)'] = [driver_programs(CALLS[k]['species']).nunique() for k in METHOD_TABLE.index]
METHOD_TABLE['programs (membership check)'] = [membership_programs(CALLS[k]['species']).nunique() for k in METHOD_TABLE.index]
METHOD_TABLE.to_csv(tables / 'methods.csv')
display(METHOD_TABLE[['method', 'family', 'species calls', 'programs (driver rule)', 'programs (membership check)',
                      'relabeled calls', 'NCDR', 'specific', '∩ C1 (primary)', 'returns']].round(2))

### 6.2 · Overlap of species calls

The heat map shows the Jaccard index between every pair of methods' species call sets. The UpSet-style
panel shows the exact intersections among five representative methods:
- whole-PT GSEA (W1) and whole-PT ORA (W2);
- S1/S2/S3 ORA (D2);
- the segment-step screen (D4);
- the primary continuous test (C1).

In [ ]:
keys = [k for k in METHOD_TABLE.index if k != 'D3']
jac = pd.DataFrame([[len(CALLS[a]['species'] & CALLS[b]['species']) / max(len(CALLS[a]['species'] | CALLS[b]['species']), 1)
                     for b in keys] for a in keys], index=keys, columns=keys)
jac.to_csv(tables / 'jaccard_species_calls.csv')

UPSET = ['W1', 'W2', 'D2', 'D4', 'C1']
membership_rows = pd.DataFrame({k: [p in CALLS[k]['species'] for p in pathways] for k in UPSET}, index=pathways)
combos = membership_rows[membership_rows.any(axis=1)].value_counts().sort_values(ascending=False)
combos.rename('pathways').to_frame().to_csv(tables / 'upset_intersections.csv')

fig = plt.figure(figsize=(180 * MM, 82 * MM))
gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.35], height_ratios=[1.4, 1], wspace=.35, hspace=.08)
ax = fig.add_subplot(gs[:, 0])
image = ax.imshow(jac.to_numpy(), cmap='Greens', vmin=0, vmax=1)
ax.set_xticks(range(len(keys)), keys, rotation=90)
ax.set_yticks(range(len(keys)), [f'{k} · {SHORT[k]}' for k in keys])
for i in range(len(keys)):
    for j in range(len(keys)):
        if i != j and jac.iloc[i, j] >= .3:
            ax.text(j, i, f'{jac.iloc[i, j]:.1f}', ha='center', va='center', fontsize=4.5,
                    color='white' if jac.iloc[i, j] > .6 else INK)
ax.set_title('a  Jaccard index of species calls', loc='left')
fig.colorbar(image, ax=ax, fraction=.04, pad=.02).ax.tick_params(labelsize=5)
top_combos = combos.head(14)
bars = fig.add_subplot(gs[0, 1])
dots = fig.add_subplot(gs[1, 1], sharex=bars)
x = np.arange(len(top_combos))
bars.bar(x, top_combos.to_numpy(), color=INK, width=.6)
for xi, v in zip(x, top_combos.to_numpy()):
    bars.text(xi, v, str(v), ha='center', va='bottom', fontsize=5)
bars.set_ylabel('pathways')
bars.set_title('b  Exact intersections of species calls (14 largest)', loc='left')
plt.setp(bars.get_xticklabels(), visible=False)
for i, k in enumerate(UPSET):
    present = np.array([combo[i] for combo in top_combos.index])
    dots.scatter(x, np.full(len(x), i), s=12, color=np.where(present, INK, LIGHT), zorder=3)
for xi, combo in zip(x, top_combos.index):
    rows_on = [i for i, flag in enumerate(combo) if flag]
    if len(rows_on) > 1:
        dots.plot([xi, xi], [min(rows_on), max(rows_on)], color=INK, lw=1)
dots.set_yticks(range(len(UPSET)), [f'{k} {SHORT[k]} ({len(CALLS[k]["species"])})' for k in UPSET])
dots.set_xticks([])
dots.invert_yaxis()
for side in ('left', 'bottom'):
    dots.spines[side].set_visible(False)
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig1_overlap.{ext}', bbox_inches='tight')
plt.show()

### 6.3 · Do the methods rank pathways alike, and agree on direction?

**Rank agreement.** Each method's per-pathway evidence for the species split is \|z\| or −log10 p.
The Spearman correlation is taken over pathways tested by both methods.

**Direction.** The signed methods are:
- W1, the sign of NES;
- W2 and D2, the DE list (up or down) with the strongest enrichment;
- D1, the sign in the segment with the strongest test;
- C1, the sign of the member-mean Δ(s) at its largest absolute value (human − mouse).

Concordance is the share of pathways called by both methods with the same sign.

In [ ]:
member_index = {p: pd.Index(genes).get_indexer(v) for p, v in gene_sets.items()}
pathway_delta = pd.DataFrame({p: curves['species']['delta'][idx].mean(axis=0) for p, idx in member_index.items()},
                             index=np.round(grid, 4)).T
peak = pathway_delta.abs().idxmax(axis=1)
DIRECTION['C1'] = pd.Series(np.sign([pathway_delta.loc[p, peak[p]] for p in pathways]), index=pathways)
evidence = pd.DataFrame(EVIDENCE)
rank_corr = evidence.corr(method='spearman')
rank_corr.to_csv(tables / 'rank_correlation_of_evidence.csv')
signed = [k for k in ('W1', 'W2', 'D1', 'D2', 'C1') if k in DIRECTION]
concordance = pd.DataFrame(index=signed, columns=signed, dtype=float)
shared = pd.DataFrame(index=signed, columns=signed, dtype=float)
for a in signed:
    for b in signed:
        both = sorted(CALLS[a]['species'] & CALLS[b]['species'])
        shared.loc[a, b] = len(both)
        concordance.loc[a, b] = (DIRECTION[a][both] == DIRECTION[b][both]).mean() if both else np.nan
concordance.to_csv(tables / 'direction_concordance.csv')
shared.to_csv(tables / 'direction_concordance_n.csv')
print('Spearman correlation of per-pathway evidence:')
display(rank_corr.round(2))
print('Direction concordance among pathways called by both (n in the second table):')
display(concordance.round(2))
display(shared.astype(int))

### 6.4 · What each family finds that the others do not

- **Specific** whole-PT and discrete methods: W2, D2 and D4.
- **Continuous reference:** the primary test, C1.
- The counts are repeated including the non-specific GSEA methods (W1, D1) and against the
  segment-step screen alone.
- For the continuous-only pathways, notebook 50's step-screen q-value shows how close the discrete
  screen came.

In [ ]:
specific_wd = CALLS['W2']['species'] | CALLS['D2']['species'] | CALLS['D4']['species']
all_wd = specific_wd | CALLS['W1']['species'] | CALLS['D1']['species']
C = CALLS['C1']['species']
misses = pd.Series({
    'C1 calls': len(C),
    'C1 not called by any specific whole-PT or discrete method (W2, D2, D4)': len(C - specific_wd),
    'C1 not called by any whole-PT or discrete method, GSEA included': len(C - all_wd),
    'C1 not called by the segment-step screen (D4)': len(C - CALLS['D4']['species']),
    'specific whole-PT or discrete calls not called by C1': len(specific_wd - C),
    'segment-step (D4) calls not called by C1': len(CALLS['D4']['species'] - C),
    'S1/S2/S3 ORA (D2) calls not called by C1': len(CALLS['D2']['species'] - C),
    'whole-PT ORA (W2) calls not called by C1': len(CALLS['W2']['species'] - C)}, name='pathways')
misses.to_csv(tables / 'misses.csv')
display(misses.to_frame())
only_c = sorted(C - CALLS['D4']['species'])
detail = pd.DataFrame({'name': NAMES[only_c], 'C1 z_joint': EVIDENCE['C1'][only_c],
                       'step-screen q (D4)': steps.q_joint.reindex(only_c),
                       'called by D2': [p in CALLS['D2']['species'] for p in only_c]}).sort_values('C1 z_joint', ascending=False)
detail.to_csv(tables / 'continuous_not_steps.csv')
display(detail.round(3))
only_d = sorted(CALLS['D4']['species'] - C)
display(pd.DataFrame({'name': NAMES[only_d], 'D4 z_joint': steps.z_joint.reindex(only_d),
                      'C1 q_joint': tests37[tests37.statistic.eq('T_spatial') & tests37.partition.eq('species')]
                      .set_index('pathway_id').q_joint.reindex(only_d)}).round(3))

## 7 · Worked examples: one pathway score, three views

Every view shows the same per-structure score, the mean-z module score of method C6:
- **whole PT:** specimen means;
- **S1/S2/S3:** specimen × segment means, bars for the species mean;
- **continuous:** a specimen-balanced species curve (6-df spline) with specimen × 12-bin means as
  points.

The pathways were chosen by the rules fixed in `plan.md` before the notebook ran:

In [ ]:
beyond = pd.read_csv(nb57 / 'per_pathway_beyond_steps.csv', index_col=0)
ranks = evidence.rank(pct=True)
rules = {}
agree = sorted(CALLS['W2']['species'] & CALLS['D2']['species'] & CALLS['D4']['species'] & C)
rules['E1 · all methods agree'] = ranks.loc[agree, ['W2', 'D2', 'D4', 'C1']].min(axis=1).sort_values(ascending=False).index[:1] if agree else []
flat = [p for p in C if steps.q_joint.get(p, 1) > .05 and p not in CALLS['D1']['species'] and p not in CALLS['D2']['species']
        and bool(beyond.loc[p, 'called test_B_coordinate_steps'])]
rules['E2 · segment means flat, curves structured'] = beyond.loc[flat, 'z_joint test_B_coordinate_steps'].sort_values(ascending=False).index[:1] if flat else []
discrete_only = sorted(CALLS['D2']['species'] - C)
rules['E3 · discrete call, no continuous call'] = EVIDENCE['D2'][discrete_only].sort_values(ascending=False).index[:1] if discrete_only else []
reversing = []
for p in C:
    d = pathway_delta.loc[p].to_numpy()
    threshold = .25 * np.abs(d).max()
    if (d >= threshold).any() and (d <= -threshold).any():
        reversing.append(p)
rules['E4 · direction depends on position'] = EVIDENCE['C1'][reversing].sort_values(ascending=False).index[:1] if reversing else []
steplike = [p for p in C & CALLS['D4']['species'] if not bool(beyond.loc[p, 'called test_B_coordinate_steps'])]
rules['E5 · step-like: continuous adds nothing'] = steps.z_joint[steplike].sort_values(ascending=False).index[:1] if steplike else []
candidates = {'E1': len(agree), 'E2': len(flat), 'E3': len(discrete_only), 'E4': len(reversing), 'E5': len(steplike)}
EXAMPLES = {}
for rule, picked in rules.items():
    if len(picked):
        EXAMPLES[rule] = picked[0]
        print(f'{rule}: {NAMES[picked[0]]} (pathways meeting the rule: {candidates[rule[:2]]})')
    else:
        print(f'{rule}: no pathway meets the rule')

In [ ]:
score_matrix = scores['mean_z']
weights = np.zeros(len(position))
for sp, names_ in SPECIMENS.items():
    for n in names_:
        rows = specimen == n
        weights[rows] = len(position) / (2 * len(names_) * rows.sum())
fine_bins = np.clip(((position - lo) / (hi - lo) * EXAMPLE_BINS).astype(int), 0, EXAMPLE_BINS - 1)
transitions = pd.read_csv(nb37 / 'tables' / 'segment_transitions_by_specimen.csv').groupby('transition').position.mean()


def species_curve(values, species_mask):
    design = make_gam_design(position[species_mask], knots)
    w = np.sqrt(weights[species_mask])
    beta, *_ = np.linalg.lstsq(w[:, None] * design, w * values[species_mask], rcond=None)
    return make_gam_design(grid, knots) @ beta


fig, axes = plt.subplots(len(EXAMPLES), 3, figsize=(180 * MM, 38 * MM * len(EXAMPLES)),
                         gridspec_kw={'width_ratios': [.55, 1, 1.8]}, squeeze=False)
example_rows = []
for r, (rule, pathway) in enumerate(EXAMPLES.items()):
    values = score_matrix[:, pathways.index(pathway)]
    ax_w, ax_s, ax_c = axes[r]
    for k, (sp, color) in enumerate((('mouse', MOUSE), ('human', HUMAN))):
        means = [values[specimen == n].mean() for n in SPECIMENS[sp]]
        ax_w.bar(k, np.mean(means), color=color, alpha=.35, width=.6)
        for n, m in zip(SPECIMENS[sp], means):
            ax_w.scatter(k, m, color=color, marker=SPECIMEN_MARKER[n], s=10, zorder=3, edgecolor='white', linewidth=.4)
        for j, seg in enumerate(('PT-S1', 'PT-S2', 'PT-S3')):
            seg_means = [values[(specimen == n) & (segment == seg)].mean() for n in SPECIMENS[sp]]
            xpos = j + (k - .5) * .38
            ax_s.bar(xpos, np.mean(seg_means), width=.36, color=color, alpha=.35)
            for n, m in zip(SPECIMENS[sp], seg_means):
                ax_s.scatter(xpos, m, color=color, marker=SPECIMEN_MARKER[n], s=10, zorder=3, edgecolor='white', linewidth=.4)
        mask = ~human if sp == 'mouse' else human
        ax_c.plot(grid, species_curve(values, mask), color=color, lw=1.4)
        for n in SPECIMENS[sp]:
            rows = specimen == n
            b = pd.Series(values[rows]).groupby(fine_bins[rows]).mean()
            centres = lo + (b.index.to_numpy() + .5) * (hi - lo) / EXAMPLE_BINS
            ax_c.scatter(centres, b.to_numpy(), color=color, marker=SPECIMEN_MARKER[n], s=7, alpha=.8,
                         edgecolor='white', linewidth=.3, zorder=3)
    for t in transitions:
        ax_c.axvline(t, color=NULL, lw=.6, ls=':')
    ax_w.set_xticks([0, 1], ['mouse', 'human'])
    ax_s.set_xticks([0, 1, 2], ['S1', 'S2', 'S3'])
    ax_c.set_xlim(lo, hi)
    ax_w.set_ylabel('pathway score (mean z)')
    calls_here = [k for k in ('W1', 'W2', 'D1', 'D2', 'D4', 'C1', 'C6') if pathway in CALLS[k]['species']]
    ax_w.set_title(f'{rule.split(" · ")[0]}  whole PT', loc='left')
    ax_s.set_title('S1 / S2 / S3', loc='left')
    ax_c.set_title(f'{NAMES[pathway][:60]} · called by {", ".join(calls_here) or "none"}', loc='left')
    if r == len(EXAMPLES) - 1:
        ax_c.set_xlabel('PT coordinate (SCF13); dotted lines: mean S1→S2 and S2→S3 transitions')
    example_rows.append({'example': rule, 'pathway': NAMES[pathway], 'called by': ', '.join(calls_here),
                         'step-screen q (D4)': steps.q_joint.get(pathway, np.nan),
                         'C1 q_joint': tests37[tests37.statistic.eq('T_spatial') & tests37.partition.eq('species')
                                               & tests37.pathway_id.eq(pathway)].q_joint.iloc[0],
                         'beyond coordinate steps called (notebook 57)': bool(beyond.loc[pathway, 'called test_B_coordinate_steps'])})
handles = [plt.Line2D([], [], color=MOUSE, lw=1.4, label='mouse'), plt.Line2D([], [], color=HUMAN, lw=1.4, label='human'),
           plt.Line2D([], [], color=INK, marker='o', lw=0, markersize=3, label='specimen 1'),
           plt.Line2D([], [], color=INK, marker='s', lw=0, markersize=3, label='specimen 2')]
axes[0, 2].legend(handles=handles, loc='upper left', ncol=4, bbox_to_anchor=(0, 1.32))
fig.tight_layout()
for ext in ('pdf', 'png'):
    fig.savefig(figures / f'fig2_worked_examples.{ext}', bbox_inches='tight')
plt.show()
EXAMPLE_TABLE = pd.DataFrame(example_rows)
EXAMPLE_TABLE.to_csv(tables / 'worked_examples.csv', index=False)
display(EXAMPLE_TABLE.round(4))

## 8 · Gene-level heatmaps by program: which genes drive the signal, and where

A pathway score averages its members, so it hides the genes that drive the call and cancels members
that change in opposite directions. These heatmaps show the genes themselves, one heatmap per program
(section 6.0), so near-duplicate pathways are not shown twice.

- **Rows.** The union of the member pathways' contributing genes (top 10% by T_spatial; all members
  when fewer than three qualify), at most 40, chosen by T_spatial. A bullet marks top-decile genes.
  The left matrix shows which member pathway, numbered in the title, contains each gene.
- **Columns.** The 12 coordinate bins. Mouse and human sit side by side. Each gene's
  specimen-averaged bin means are z-scored within each species, which removes the level offset.
  Dotted lines mark each species' mean S1→S2 and S2→S3 transitions; a dot marks the peak bin.
- **Row order.** By mouse peak bin, then human peak bin, so waves of genes switching on or off are
  visible.
- **Right of the heatmap:**
  - notebook 43 zonation class;
  - log10 T_spatial;
  - whole-PT DESeq2 log2 human/mouse (H/M), so the offset is not lost.
- **Below:** each specimen's own z-scored bins in the same order, the replication check.

The notebook shows the programs that contain the worked examples (among the 60 C1 calls; E3 is not a
C1 call and is shown alone). The gallery `figures/gene_heatmaps_primary22.pdf` has one page per
program of the 22 primary pathways. Display rules were fixed in `plan.md` before this section ran.

In [ ]:
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colors import LinearSegmentedColormap

MAX_ROWS, Z_CLIP = 40, 2.5
HEAT = LinearSegmentedColormap.from_list('teal_grey_brown', ['#01665E', '#5AB4AC', '#E6E6E6', '#D8B365', '#8C510A'])
CLASS_COLORS = {'conserved': '#009E73', 'reversal': '#CC79A7', 'mouse-only': MOUSE, 'human-only': HUMAN,
                'indeterminate': '#E3E3E3', 'neither': '#B8B8B8'}
SHORT_SPECIMEN = {'Ctrl1A2': 'M1', 'Ctrl1A4': 'M2', 'HUK1_COR1': 'H1', 'HUK1_MED1': 'H2'}


def specimen_bin_means():
    """Mean log-normalised expression per specimen x coordinate bin (genes x bins per specimen)."""
    out = {}
    for n in ALL_SPECIMENS:
        rows = np.flatnonzero(specimen == n)
        indicator = sparse.csr_matrix((np.ones(len(rows)), (fine_bins[rows], np.arange(len(rows)))),
                                      shape=(EXAMPLE_BINS, len(rows)))
        sums = np.asarray((indicator @ inputs['Y'][rows]).todense())
        out[n] = (sums / np.asarray(indicator.sum(axis=1))).T
    return out


bin_means = cached_payload('specimen_bin_means', specimen_bin_means, root=cache, params={'bins': EXAMPLE_BINS},
                           inputs=input_key, code=code)
species_means = {sp: np.mean([bin_means[n] for n in SPECIMENS[sp]], axis=0) for sp in SPECIMENS}


def zscore_rows(values):
    centred = values - values.mean(axis=1, keepdims=True)
    sd = values.std(axis=1, keepdims=True)
    return np.where(sd > 1e-9, centred / np.where(sd > 1e-9, sd, 1), 0.)


species_z = {sp: zscore_rows(v) for sp, v in species_means.items()}
specimen_z = {n: zscore_rows(v) for n, v in bin_means.items()}
gene_position = pd.Series(np.arange(len(genes)), index=genes)
t_spatial = gene37
top_decile = set(t_spatial.nlargest(int(round(TOP_SHARE * len(genes)))).index)
whole_lfc = deseq[deseq.partition.eq('species') & deseq.scope.eq('whole_PT')].set_index('gene').log2FoldChange
zonation = pd.read_csv(results_root / 'pt_revision_classes' / 'reviewed' / 'tables' / 'gene_classes_symmetric.csv',
                       index_col=0)['union S2-S1 + S3c-early']
species_transitions = pd.read_csv(nb37 / 'tables' / 'segment_transitions_by_specimen.csv').groupby(
    ['species', 'transition']).position.mean()
bin_edges = np.linspace(lo, hi, EXAMPLE_BINS + 1)


def program_rows(member_pathways):
    """Union of the members' contributing genes (fallback: all members), capped at the top 40 by T_spatial."""
    union = []
    for p in member_pathways:
        members = [g for g in gene_sets[p] if g in gene_position.index]
        chosen = [g for g in members if g in contributing]
        union.extend(chosen if len(chosen) >= 3 else members)
    union = sorted(set(union))
    shown = union if len(union) <= MAX_ROWS else list(t_spatial[union].nlargest(MAX_ROWS).index)
    idx = gene_position[shown].to_numpy()
    peaks = pd.DataFrame({'mouse': species_z['mouse'][idx].argmax(axis=1), 'human': species_z['human'][idx].argmax(axis=1),
                          'gene': shown})
    order = peaks.sort_values(['mouse', 'human', 'gene']).index.to_numpy()
    return [shown[i] for i in order], len(union)


def draw_program_heatmap(member_pathways, title):
    member_pathways = sorted(member_pathways, key=lambda p: -EVIDENCE['C1'].get(p, 0))
    rows, n_union = program_rows(member_pathways)
    idx = gene_position[rows].to_numpy()
    n, k = len(rows), len(member_pathways)
    height_mm = 40 + n * 2.4 * 1.6 + 3 * k
    fig = plt.figure(figsize=(180 * MM, height_mm * MM))
    widths = [max(k, 1) * .9, 12, 1.2, 12, 1.2, 2.4, 2.4]
    outer = fig.add_gridspec(2, 1, height_ratios=[1, .6], hspace=.16, top=1 - (14 + 3 * k) / height_mm, bottom=.04,
                             left=.12, right=.98)
    top = outer[0].subgridspec(1, 7, width_ratios=widths, wspace=.06)
    bottom = outer[1].subgridspec(1, 7, width_ratios=widths, wspace=.06)
    mx = fig.add_subplot(top[0, 0])
    matrix = np.array([[g in set(gene_sets[p]) for p in member_pathways] for g in rows], float)
    yy, xx = np.nonzero(matrix)
    mx.scatter(xx, yy, s=4, color=INK, linewidths=0)
    mx.set_xlim(-.5, k - .5)
    mx.set_ylim(n - .5, -.5)
    mx.set_xticks(range(k), [str(i + 1) for i in range(k)], fontsize=5)
    mx.xaxis.tick_top()
    mx.set_yticks(range(n), [('• ' if g in top_decile else '') + g for g in rows], fontsize=5)
    for side in ('right', 'bottom'):
        mx.spines[side].set_visible(False)
    mx.set_title('in pathway', loc='left', fontsize=5.5, pad=8)
    for col, sp in ((1, 'mouse'), (3, 'human')):
        ax = fig.add_subplot(top[0, col])
        image = ax.imshow(np.clip(species_z[sp][idx], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP,
                          aspect='auto', interpolation='nearest')
        ax.scatter(species_z[sp][idx].argmax(axis=1), np.arange(n), s=3, color=INK, zorder=3, linewidths=0)
        for t in ('S1→S2', 'S2→S3'):
            ax.axvline((species_transitions[(sp, t)] - lo) / (hi - lo) * EXAMPLE_BINS - .5, color=INK, lw=.6, ls=':')
        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'{sp}: specimen-averaged, z within species', loc='left', color=MOUSE if sp == 'mouse' else HUMAN)
        sub = bottom[0, col].subgridspec(1, 2, wspace=.08)
        for j, name in enumerate(SPECIMENS[sp]):
            bx = fig.add_subplot(sub[0, j])
            bx.imshow(np.clip(specimen_z[name][idx], -Z_CLIP, Z_CLIP), cmap=HEAT, vmin=-Z_CLIP, vmax=Z_CLIP,
                      aspect='auto', interpolation='nearest')
            for t in ('S1→S2', 'S2→S3'):
                bx.axvline((species_transitions[(sp, t)] - lo) / (hi - lo) * EXAMPLE_BINS - .5, color=INK, lw=.5, ls=':')
            bx.set_xticks([0, EXAMPLE_BINS - 1], ['start', 'end'], fontsize=5)
            bx.set_yticks([])
            bx.set_title(f'{SHORT_SPECIMEN[name]} ({name})', loc='left', fontsize=5.5, fontweight='normal')
    cx = fig.add_subplot(top[0, 4])
    for i, g in enumerate(rows):
        c = zonation.get(g, np.nan)
        cx.add_patch(plt.Rectangle((0, i - .4), 1, .8, color=CLASS_COLORS.get(c, '#FFFFFF'),
                                   ec=LIGHT if c not in CLASS_COLORS else 'none', lw=.3))
    cx.set_xlim(0, 1)
    cx.set_ylim(n - .5, -.5)
    cx.axis('off')
    cx.set_title('class', loc='left', fontsize=5.5)
    tx = fig.add_subplot(top[0, 5])
    tx.barh(np.arange(n), np.log10(t_spatial[rows].clip(lower=1)), color=ACCENT, height=.7)
    tx.set_ylim(n - .5, -.5)
    tx.set_yticks([])
    tx.set_title('log10 T', loc='left', fontsize=5.5)
    tx.tick_params(labelsize=5)
    lx = fig.add_subplot(top[0, 6])
    lfc = whole_lfc.reindex(rows).clip(-6, 6).fillna(0).to_numpy()
    lx.barh(np.arange(n), lfc, color=np.where(lfc > 0, HUMAN, MOUSE), height=.7)
    lx.axvline(0, color=INK, lw=.5)
    lx.set_xlim(-6.5, 6.5)
    lx.set_ylim(n - .5, -.5)
    lx.set_yticks([])
    lx.set_title('log2 H/M', loc='left', fontsize=5.5)
    lx.tick_params(labelsize=5)
    cax = fig.add_subplot(bottom[0, 4:])
    cax.axis('off')
    bar = fig.colorbar(image, ax=cax, orientation='horizontal', fraction=.45, aspect=10, pad=0)
    bar.set_label('z within species (clipped ±2.5)', fontsize=5)
    bar.ax.tick_params(labelsize=5)
    legend = [plt.Rectangle((0, 0), 1, 1, color=v) for v in CLASS_COLORS.values()] + \
        [plt.Rectangle((0, 0), 1, 1, facecolor='white', edgecolor=LIGHT)]
    cax.legend(legend, list(CLASS_COLORS) + ['not classified'], loc='upper left', fontsize=4.5, ncol=2,
               handlelength=1, borderaxespad=0)
    called = [m for m in ('W1', 'W2', 'D1', 'D2', 'D4', 'C1') if any(p in CALLS[m]['species'] for p in member_pathways)]
    header = f'{title} · {n} of {n_union} genes shown · members called by {", ".join(called) or "none"}'
    listing = '; '.join(f'{i + 1} {NAMES[p]}' for i, p in enumerate(member_pathways))
    fig.text(.01, 1 - 4 / height_mm, header, ha='left', va='top', fontsize=6.5, fontweight='bold')
    fig.text(.01, 1 - 8 / height_mm, listing, ha='left', va='top', fontsize=5, wrap=True, color=MUTED)
    return fig


def program_of(pathway, labels):
    return sorted(labels.index[labels.eq(labels[pathway])]) if pathway in labels.index else [pathway]


c1_names = program_names(c1_drivers, EVIDENCE['C1'])
for rule, pathway in EXAMPLES.items():
    members = program_of(pathway, c1_drivers)
    name = c1_names[pathway] if pathway in c1_names.index else NAMES[pathway]
    kind = f'program "{name}" ({len(members)} pathways among the C1 calls)' if pathway in c1_drivers.index else 'one pathway (not a C1 call)'
    fig = draw_program_heatmap(members, f'{rule.split(" · ")[0]} · {kind}')
    for ext in ('pdf', 'png'):
        fig.savefig(figures / f'fig3_gene_heatmap_{rule.split(" · ")[0]}.{ext}')
    plt.show()

primary_names = program_names(primary_drivers, EVIDENCE['C1'])
gallery = []
with PdfPages(figures / 'gene_heatmaps_primary22.pdf') as pdf:
    for program, block in primary_drivers.groupby(primary_drivers):
        members = sorted(block.index)
        fig = draw_program_heatmap(members, f'Program {program}: {primary_names[members[0]]} ({len(members)} of the primary 22)')
        pdf.savefig(fig)
        plt.close(fig)
        gallery.append({'program': program, 'name': primary_names[members[0]], 'pathways': len(members),
                        'members': '; '.join(NAMES[members])})
GALLERY = pd.DataFrame(gallery)
GALLERY.to_csv(tables / 'primary22_program_gallery.csv', index=False)
display(GALLERY)
print(f'Gallery: {len(GALLERY)} programs of the primary 22, one page each, in figures/gene_heatmaps_primary22.pdf')

## 9 · Deduplicating near-identical library entries before testing

Reactome, KEGG and Hallmark contain near-copies of the same gene set. Section 6.0 groups them after
testing. Here they are merged **before** testing, using membership alone (no data): two sets are
linked when their member Jaccard is ≥ 0.7, and each connected component becomes one set, the union
of its members.

The primary method (T_spatial joint test) and two comparators are rerun on the collapsed library, each
under the species split and both relabelings:
- the coordinate-free segment-step screen (discrete);
- whole-PT DE + ORA (average-level).

The decision rule, frozen in `plan.md` (addendum 3), says whether deduplication changes the
substantive result or only tidies the counts. Deduplication is not adopted as primary here.

In [ ]:
from scipy.sparse.csgraph import connected_components
from scipy.stats import norm
from pseudospace.pathway_remodeling import (correlation_adjusted_rank_tests, fit_nested_trajectories,
                                            residual_pathway_correlations)
from pseudospace.specimen_null import matched_auc_null, summary_partial_f

DEDUP_JACCARD, SENSITIVITY_JACCARD = .7, (.5, .9)
member_sets = {p: set(gene_sets[p]) for p in pathways}
size = pd.Series({p: len(v) for p, v in member_sets.items()})
binary = membership(gene_sets, genes)
intersection = (binary @ binary.T).toarray()
union_size = size.to_numpy()[:, None] + size.to_numpy()[None, :] - intersection
jaccard = intersection / union_size


def components_at(threshold):
    _, labels = connected_components(sparse.csr_matrix(jaccard >= threshold), directed=False)
    return pd.Series(labels, index=pathways)


dedup_counts = pd.Series({t: components_at(t).nunique() for t in (*SENSITIVITY_JACCARD[:1], DEDUP_JACCARD, *SENSITIVITY_JACCARD[1:])},
                         name='sets after collapse').rename_axis('Jaccard threshold')
display(dedup_counts.to_frame().assign(**{'sets before': len(pathways)}))
component = components_at(DEDUP_JACCARD)
COLLAPSED, MEMBERS_OF = {}, {}
for label, block in component.groupby(component):
    ids = sorted(block.index)
    lead = sorted(ids, key=lambda p: (-size[p], p))[0]
    key = lead if len(ids) == 1 else f'{lead} (+{len(ids) - 1})'
    COLLAPSED[key] = sorted(set().union(*[member_sets[p] for p in ids]))
    MEMBERS_OF[key] = ids
component_of = {p: k for k, ids in MEMBERS_OF.items() for p in ids}
largest = sorted(MEMBERS_OF, key=lambda k: -len(MEMBERS_OF[k]))[:10]
LARGEST = pd.DataFrame({'collapsed set': [NAMES[MEMBERS_OF[k][0]] if len(MEMBERS_OF[k]) == 1 else
                                          NAMES[sorted(MEMBERS_OF[k], key=lambda p: (-size[p], p))[0]] for k in largest],
                        'pathways merged': [len(MEMBERS_OF[k]) for k in largest],
                        'union size': [len(COLLAPSED[k]) for k in largest],
                        'members': ['; '.join(NAMES[MEMBERS_OF[k]]) for k in largest]})
LARGEST.to_csv(tables / 'dedup_largest_components.csv', index=False)
pd.DataFrame([{'collapsed set': k, 'pathways merged': len(v), 'union size': len(COLLAPSED[k]), 'members': '; '.join(v)}
              for k, v in MEMBERS_OF.items()]).to_csv(tables / 'dedup_components.csv', index=False)
print(f'{len(pathways)} sets → {len(COLLAPSED)} after collapsing at Jaccard ≥ {DEDUP_JACCARD}; '
      f'{sum(len(v) > 1 for v in MEMBERS_OF.values())} components merge two or more sets')
display(LARGEST)

### 9.1 · The three methods on the collapsed library

- **T_spatial joint test.** Notebook 37's gene scores, strata and 9,999 matched sets. The VIF is
  recomputed for the collapsed sets with notebook 37's recipe; a guard checks that the recipe
  reproduces notebook 37's VIF on the original library.
- **Segment-step screen.** Recomputed from notebook 45's cell summaries for our cohort, with exact
  matched moments and a segment-centred VIF. A guard checks that it reproduces notebook 50 (50 calls,
  0 and 0 relabeled) on the original library.
- **Whole-PT ORA.** From the saved DESeq2 gene lists.

In [ ]:


def tspatial_vif(sets):
    def compute():
        fit = fit_nested_trajectories(inputs['Y'], position, human, specimen, grid, basis_df=6, return_residuals=True)
        out = {}
        for name, family in (('original', gene_sets), ('collapsed', COLLAPSED)):
            corr = residual_pathway_correlations(fit['residuals'], specimen, genes, family)
            camera = correlation_adjusted_rank_tests(gene37, family, corr).set_index('pathway_id').variance_inflation
            out[name] = camera.reindex(list(family)).to_numpy(float)
        return out
    return cached_payload('dedup_tspatial_vif', compute, root=cache, params={'jaccard': DEDUP_JACCARD},
                          inputs={**input_key, 'collapsed': COLLAPSED}, code=code)


vif_payload = tspatial_vif(COLLAPSED)
np.testing.assert_allclose(vif_payload['original'], VIF.to_numpy(), rtol=1e-6)   # Guard: notebook 37's VIF recipe
VIF_COLLAPSED = pd.Series(vif_payload['collapsed'], index=list(COLLAPSED))
gene_scores37 = {p: np.load(sorted(glob.glob(str(nb37 / 'stage_cache' /
                                                  f"gene_scores_{p.replace(':', '_').replace('+', '_')}__*.npz")))[-1])['T_spatial']
                 for p in PARTS}
np.testing.assert_allclose(gene_scores37['species'], gene37.to_numpy(), rtol=1e-6)   # Guard


def tspatial_calls(sets, vif, label):
    out = {}
    for p in PARTS:
        frame = cached_frame(f"dedup_tspatial_{label}_{p.replace(':', '_').replace('+', '_')}",
                             lambda p=p: matched_pathway_tests(pd.DataFrame({'T_spatial': gene_scores37[p]}, index=genes),
                                                               sets, strata, n_null=N_NULL, seed=SEED),
                             root=cache, params={'n_null': N_NULL, 'seed': SEED},
                             inputs={'scores': gene_scores37[p], 'sets': sets, 'strata': strata}, code=code)
        tests = joint_matched_test(frame, vif, keys=('statistic',)).set_index('pathway_id')
        out[p] = tests
    return out


ts_collapsed = tspatial_calls(COLLAPSED, VIF_COLLAPSED, 'collapsed')
ts_original = {p: tests37[tests37.statistic.eq('T_spatial') & tests37.partition.eq(p)].set_index('pathway_id') for p in PARTS}

# Segment-step screen (notebook 45's machinery for our cohort).
pool = newest(results_root / 'pt_revision_method' / 'stage_cache', 'pool_our_cohort')
pool_genes = list(pool['genes'].astype(str))
assert pool_genes == list(genes), 'notebook 45 pool genes differ from the universe'
pool_strata = pool['strata']
cohort_donors = ALL_SPECIMENS
species_group = {d: float(d in SPECIMENS['human']) for d in cohort_donors}
step_designs = {'species': (species_group, None)}
for s_ in SWAPS:
    step_designs[s_] = ({d: float(d in partitions[s_]) for d in cohort_donors}, species_group)


def step_vif(sets):
    """Notebook 45's pool VIF: units centred within specimen x segment, mean member correlation per specimen."""
    def compute():
        x = inputs['Y'].toarray()
        corr = []
        for name in cohort_donors:
            rows = x[specimen == name].copy()
            seg = segment[specimen == name]
            for k in np.unique(seg):
                rows[seg == k] -= rows[seg == k].mean(axis=0)
            corr.append(residual_pathway_correlations(rows, np.repeat(name, len(rows)), genes, sets))
        reference = pd.Series(np.mean([[np.asarray(inputs['Y'][specimen == n].mean(axis=0)).ravel() for n in SPECIMENS[sp]]
                                       for sp in SPECIMENS], axis=(0, 1)), index=genes)
        camera = correlation_adjusted_rank_tests(reference, sets, pd.concat(corr, ignore_index=True))
        return {'vif': camera.set_index('pathway_id').variance_inflation.reindex(list(sets)).to_numpy(float)}
    return cached_payload(f'dedup_step_vif_{len(sets)}', compute, root=cache, params={'sets': len(sets)},
                          inputs={**input_key, 'sets': sets}, code=code)['vif']


def step_calls(sets, vif):
    out = {}
    for p, (group, nuisance) in step_designs.items():
        f = summary_partial_f(pool, cohort_donors, group, nuisance)
        tests = matched_auc_null(pd.DataFrame({'T_steps': f['T_steps']}, index=genes), sets, pool_strata).set_index('pathway_id')
        tests['z_joint'] = tests.z_matched / np.sqrt(pd.Series(vif, index=list(sets)))
        tests['q_joint'] = multipletests(norm.sf(tests.z_joint), method='fdr_bh')[1]
        out[p] = tests
    return out


# Notebook 45 built its sets from notebook 12's universe; they equal notebook 37's tested members except one
# gene in one pathway (Nat2 in Hallmark UV Response Up), so the guards exclude that pathway.
pool_vif = pd.Series(pool['vif'], index=pool['pathways'].astype(str))
same_members = [p for p in pathways if p in pool_vif.index]
recipe_vif = pd.Series(step_vif(gene_sets), index=pathways)
differs = [p for p in pathways if not np.isclose(recipe_vif[p], pool_vif[p], rtol=1e-6)]
assert len(differs) <= 1, differs                                                    # Guard: notebook 45's VIF recipe
steps_original = step_calls(gene_sets, pool_vif.reindex(pathways).to_numpy())
called_step = {p: set(t.index[t.effect.gt(0) & t.q_joint.le(ALPHA)]) for p, t in steps_original.items()}
assert called_step['species'] == set(steps.index[steps.called.astype(bool)])          # Guard: notebook 50
assert [len(called_step[s_]) for s_ in SWAPS] == [0, 0]
check = [p for p in steps.index if p not in differs and p != 'MSigDB_Hallmark_2020::UV Response Up']
np.testing.assert_allclose(steps_original['species'].z_joint.reindex(check), steps.z_joint.reindex(check), rtol=1e-6, atol=1e-8)
steps_collapsed = step_calls(COLLAPSED, step_vif(COLLAPSED))


def ora_calls_family(sets):
    out = {}
    for p in PARTS:
        rows = deseq[deseq.partition.eq(p) & deseq.scope.eq('whole_PT')].dropna(subset=['stat']).set_index('gene')
        matrix = membership(sets, list(rows.index))
        cols = [ora_pvalues(rows.padj.le(ALPHA).to_numpy() & (d * rows.log2FoldChange).ge(LFC).to_numpy(), matrix) for d in (1, -1)]
        pv = np.column_stack(cols)
        q = multipletests(pv.ravel(), method='fdr_bh')[1].reshape(pv.shape)
        out[p] = pd.DataFrame({'q_joint': q.min(axis=1), 'effect': 1., 'z_joint': norm.isf(np.clip(pv.min(axis=1), 1e-300, 1))},
                              index=list(sets))
    return out


ora_original, ora_collapsed = ora_calls_family(gene_sets), ora_calls_family(COLLAPSED)
assert set(ora_original['species'].index[ora_original['species'].q_joint.le(ALPHA)]) == CALLS['W2']['species']   # Guard

### 9.2 · Original against collapsed: calls, specificity, kept / lost / new, and programs

In [ ]:


def call_sets(tests):
    return {p: set(t.index[t.effect.gt(0) & t.q_joint.le(ALPHA)]) for p, t in tests.items()}


comparison_rows, verdicts = [], {}
FAMILIES = {'T_spatial joint (C1)': (ts_original, ts_collapsed), 'segment steps (D4)': (steps_original, steps_collapsed),
            'whole-PT ORA (W2)': (ora_original, ora_collapsed)}
called_both = {}
for label, (orig, coll) in FAMILIES.items():
    for version, tests, n_sets in (('original', orig, len(pathways)), ('collapsed', coll, len(COLLAPSED))):
        calls = call_sets(tests)
        called_both[(label, version)] = calls
        relabels = [len(calls[s_]) for s_ in SWAPS]
        ncdr = np.mean(relabels) / len(calls['species']) if calls['species'] else np.nan
        specific = bool(ncdr < NCDR_MAX and max(relabels) < NULL_RATE * n_sets)
        verdicts[(label, version)] = specific
        comparison_rows.append({'method': label, 'library': version, 'sets tested': n_sets, 'species calls': len(calls['species']),
                                'relabeled calls': ', '.join(map(str, relabels)), 'NCDR': ncdr, 'specific': specific})
DEDUP_COMPARISON = pd.DataFrame(comparison_rows)
DEDUP_COMPARISON.to_csv(tables / 'dedup_calls_and_specificity.csv', index=False)
display(DEDUP_COMPARISON.round(3))

original_c1 = called_both[('T_spatial joint (C1)', 'original')]['species']
collapsed_c1 = called_both[('T_spatial joint (C1)', 'collapsed')]['species']
contributing_rank = gene37.rank(pct=True)
mapping = []
for p in sorted(original_c1):
    k = component_of[p]
    added = sorted(set(COLLAPSED[k]) - member_sets[p])
    mapping.append({'pathway': NAMES[p], 'collapsed set': k, 'status': 'kept' if k in collapsed_c1 else 'lost',
                    'original z_joint': ts_original['species'].loc[p, 'z_joint'],
                    'collapsed z_joint': ts_collapsed['species'].loc[k, 'z_joint'],
                    'members': len(member_sets[p]), 'union': len(COLLAPSED[k]),
                    'members mean T percentile': contributing_rank[list(member_sets[p])].mean(),
                    'added genes mean T percentile': contributing_rank[added].mean() if added else np.nan})
MAPPING = pd.DataFrame(mapping)
MAPPING['union dilution'] = (MAPPING.status.eq('lost') & MAPPING['added genes mean T percentile'].lt(MAPPING['members mean T percentile'])
                             & MAPPING['collapsed z_joint'].lt(MAPPING['original z_joint']))
new_sets = sorted(k for k in collapsed_c1 if not set(MEMBERS_OF[k]) & original_c1)
NEW = pd.DataFrame({'collapsed set': new_sets, 'members': ['; '.join(NAMES[MEMBERS_OF[k]]) for k in new_sets],
                    'collapsed z_joint': [ts_collapsed['species'].loc[k, 'z_joint'] for k in new_sets],
                    'best original member z_joint': [ts_original['species'].reindex(MEMBERS_OF[k]).z_joint.max() for k in new_sets]})
MAPPING.to_csv(tables / 'dedup_c1_kept_lost.csv', index=False)
NEW.to_csv(tables / 'dedup_c1_new.csv', index=False)
print('Original C1 calls:', MAPPING.status.value_counts().to_dict(), '· new collapsed calls:', len(new_sets))
display(MAPPING[MAPPING.status.eq('lost')].round(3))
display(NEW.round(3))

# Programs both ways, order-invariant driver rule.
programs_original = driver_programs(original_c1)
programs_collapsed = overlap_programs(COLLAPSED, sorted(collapsed_c1), contributing=contributing,
                                      cut=PROGRAM_CUT).set_index('pathway_id').program
lost_programs = [g for g, block in programs_original.groupby(programs_original)
                 if not any(component_of[p] in collapsed_c1 for p in block.index)]
gained_programs = [g for g, block in programs_collapsed.groupby(programs_collapsed)
                   if not any(set(MEMBERS_OF[k]) & original_c1 for k in block.index)]
change = abs(programs_collapsed.nunique() - programs_original.nunique()) / programs_original.nunique()
spec_changed = [label for label in FAMILIES if verdicts[(label, 'original')] != verdicts[(label, 'collapsed')]]
substantive = change > .2 or lost_programs or gained_programs or spec_changed
DEDUP_DECISION = pd.Series({
    'C1 programs, original library': programs_original.nunique(),
    'C1 programs, collapsed library': programs_collapsed.nunique(),
    'relative change in program count': change,
    'original programs lost': len(lost_programs), 'programs gained': len(gained_programs),
    'methods whose specificity verdict changed': ', '.join(spec_changed) or 'none',
    'decision (plan addendum 3)': 'deduplication changes the substantive result' if substantive
    else 'deduplication only tidies the counts'}, name='value')
DEDUP_DECISION.to_csv(tables / 'dedup_decision.csv')
display(DEDUP_DECISION.to_frame())

**Reading.**
- **The frozen rule says deduplication changes the substantive result.** Rule (ii) fires: four
  programs are gained. C1's program count goes from 25 to 30, a 20% change, which is not above the
  rule (i) threshold.
- **The cause is the number of tests, not the merging.**
  - All 60 original calls are kept and no program is lost.
  - The 6 new calls are single, unmerged sets whose z barely moves (2.78–2.85 before and after).
    They cross the BH threshold because 1,160 sets are tested instead of 1,513.
  - Specificity is unchanged for all three methods.
- **Where merging matters most.** The average-level whole-PT ORA list shrinks from 80 to 44, so its
  calls were the most redundant.
- **A caveat on chaining.** Merging at Jaccard ≥ 0.7 can chain unrelated sets: the largest component
  joins 67 cell-cycle and proteasome entries into one set of 429 genes. A pre-test merge would need a
  size cap or a stricter threshold before it could be adopted.

## 10 · Which method for which question

The numbers are from the run above (species calls; the two relabeling counts).

| Question | Method to use | Why | What it cannot do |
|---|---|---|---|
| Does a program differ **on average** between species? | Whole-PT DE + ORA (W2) | Specific under relabeling (80; 0, 1) and in notebook 56's power-matched external nulls | 60% of the calls depend on genes whose probe counts differ between panels (notebook 56); no position |
| Does it differ **within a segment**? | S1/S2/S3 DE + ORA (D2) | Specific (118; 0, 1); gives the segment and direction | Mostly a within-segment offset question, with the same probe exposure; 92 of its calls are not positional at all |
| Does the **S1 → S2 → S3 profile** differ? | Segment-step screen (D4) | Specific (50; 0, 0), needs no coordinate, and is calibrated on external donors | Three levels only |
| Does the **positional profile** differ, and where? | T_spatial joint test (C1, primary) | Specific (60; 2, 0); returns D(s), the peak and the direction there | — |

**Methods to avoid for the species question:**
- GSEA on DESeq2 statistics (W1, D1) calls more pathways under relabeling than for species.
- Comparing per-species association lists (C4) is the "difference in significance" design: 74 calls
  against 61 and 66 relabeled.
- Curve modules (C7) are not specific: 44 against 23 and 12.
- Start versus end (C5) is specific but nearly powerless: 2 calls.
- Specimen-level activity curves (C6) are specific but call 85–90% of all pathways, so they rank
  nothing.

**Where continuous adds nothing.**
- **Average differences.** 92 S1/S2/S3-ORA and 61 whole-PT-ORA calls are offsets that C1 does not
  test (Oxidative phosphorylation, example E3).
- **Step-like programs.** In example E5 the curve says what the segment bars say.
- **The call list itself.** C1 agrees with the coordinate-free step screen (Jaccard 0.75; rank
  correlation 0.94) and with six coordinate bins (0.76; 0.98). Of C1's 60 calls, 13 are missed by the
  step screen, and all 13 have step q between 0.058 and 0.136. The step screen calls 3 that C1 misses.
- **Programs.** Counting programs instead of pathways (section 6.0) removes the remaining difference:
  the step screen (50 pathways), C1 (60) and six bins (49) each collapse to 25 programs. Whole-PT ORA
  collapses from 80 to 23, and S1/S2/S3 ORA from 118 to 43.
- **Deduplicating the library before testing (section 9)** changes little. 1,513 sets become 1,160.
  - C1 keeps all 60 of its calls and gains 6, all single sets that cross the threshold only because
    fewer sets are tested.
  - The step screen goes from 50 to 49 calls, whole-PT ORA from 80 to 44.
  - No specificity verdict changes.

**Where continuous adds something.** It does not add many pathways; it adds a reading of each one:
- where the species difference sits inside segments, as in E2, where mouse declines from late S2 into
  S3 while human stays flat and the step screen narrowly misses (q 0.058);
- where the direction changes along the PT, as in E4, with a human-high S1 and a mouse-high
  interior S2 peak, both replicated by the specimen bin means.

This notebook takes SCF13 as exact. With the real coordinate, notebooks 57 and 58 show that
within-segment readings need the step nulls and replication checks before they are claimed.

## 11 · Run record

In [ ]:
record = {'notebook_logic_version': NOTEBOOK_LOGIC_VERSION, 'n_structures': int(len(position)), 'n_genes': len(genes),
          'n_pathways': len(pathways), 'methods': METHOD_TABLE.drop(columns=['question']).reset_index().to_dict(orient='records'),
          'misses': misses.to_dict(), 'examples': {r: NAMES[p] for r, p in EXAMPLES.items()}}
(output / 'run_record.json').write_text(json.dumps(record, indent=2, default=str))